# Tarea 1, Parte 2. Del LLM al clasificador

**Pregunta guía.** ¿Se pueden convertir titulares peruanos en un clasificador de noticias?

En esta parte se comparan tres formas de clasificar los 375 titulares de El Comercio recolectados en la Parte 1.

1. Un modelo **zero-shot** que nunca vio estos datos y sirve como línea base.
2. Un **LLM local** de la familia Qwen2.5 de la clase 1, que etiqueta todo el dataset.
3. Un **Transformer ajustado** con fine-tuning, DistilBERT multilingüe, entrenado como en la clase 2.

In [1]:
from datetime import date

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score

pd.set_option("display.max_colwidth", 120)

noticias = pd.read_csv("datos/noticias.csv")
print(noticias.shape)
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

(375, 7)
CUDA disponible: True
GPU: NVIDIA GeForce RTX 2060


Las 4 categorías de la tarea y su nombre en castellano, que es como se las presentamos a los modelos.

In [2]:
ETIQUETAS = ["politica", "economia", "deportes", "ciencia_tecnologia"]
NOMBRES = {"politica": "política", "economia": "economía",
           "deportes": "deportes", "ciencia_tecnologia": "ciencia y tecnología"}

noticias["etiqueta"].value_counts()

etiqueta
deportes              98
ciencia_tecnologia    96
politica              93
economia              88
Name: count, dtype: int64

## 1. Línea base zero-shot

Usamos `pipeline("zero-shot-classification")` con `joeddav/xlm-roberta-large-xnli`, el mismo modelo del `Lab_Roberta` de la sesión 09. El modelo recibe el titular y la lista de categorías posibles y elige la más probable, sin haber sido entrenado con ningún titular nuestro.

Evaluamos 40 titulares al azar con semilla fija.

In [3]:
from transformers import pipeline

zero_shot = pipeline("zero-shot-classification", model="joeddav/xlm-roberta-large-xnli",
                     device=0 if torch.cuda.is_available() else -1)

candidatas = list(NOMBRES.values())
de_nombre_a_etiqueta = {v: k for k, v in NOMBRES.items()}

zero_shot(noticias.loc[0, "titular"], candidate_labels=candidatas)

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'sequence': 'Keiko Fujimori ratifica su respaldo al ministro César Astudillo y al jefe de la PNP Fredy López',
 'labels': ['política', 'deportes', 'economía', 'ciencia y tecnología'],
 'scores': [0.5263925790786743,
  0.18466556072235107,
  0.15064305067062378,
  0.1382988542318344]}

In [4]:
muestra_zs = noticias.sample(40, random_state=42)

pred_zs = []
for titular in muestra_zs["titular"]:
    resultado = zero_shot(titular, candidate_labels=candidatas)
    pred_zs.append(de_nombre_a_etiqueta[resultado["labels"][0]])

muestra_zs["pred_zero_shot"] = pred_zs

acc_zs = accuracy_score(muestra_zs["etiqueta"], muestra_zs["pred_zero_shot"])
f1_zs = f1_score(muestra_zs["etiqueta"], muestra_zs["pred_zero_shot"], average="macro")
print(f"Zero-shot en 40 titulares · accuracy {acc_zs:.3f} · F1 macro {f1_zs:.3f}")

[transformers] You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Zero-shot en 40 titulares · accuracy 0.750 · F1 macro 0.708


In [5]:
muestra_zs.loc[muestra_zs["etiqueta"] != muestra_zs["pred_zero_shot"],
               ["titular", "etiqueta", "pred_zero_shot"]]

,titular,etiqueta,pred_zero_shot
167,"BTS en Lima: cómo 118 mil fans pueden mover hasta S/200 millones entre merchandising, restaurantes y más en el conc...",economia,deportes
316,"Fallece Margaret Hamilton, pionera del programa Apolo y en la llegada del hombre a la Luna",ciencia_tecnologia,deportes
119,"Eduardo Grisolle, CEO de 121: “El modelo de las agencias de holding está cumpliendo un ciclo,es el momento de las ag...",economia,politica
280,León XIV fue operado de cáncer de pulmón: qué implica la extirpación de un lóbulo y cómo es la recuperación,ciencia_tecnologia,politica
234,Tchouaméni respalda a Mbappé para el Balón de Oro 2026: “Lo merece”,deportes,politica
274,Carlos Gonzales tras el triunfo de Cristal: “traer a Julio César Uribe y Roberto Mosquera ha sido para devolverle el...,deportes,politica
238,Alianza Lima lanza medalla por sus 125 años: cuánto cuesta y cómo es la edición limitada,deportes,economia
323,Día Mundial del Huevo: solo el 4% de las gallinas utilizadas en la industria peruana vive libre de jaulas,ciencia_tecnologia,economia
172,"Locura por BTS en Lima: preventa supera los 521 mil fans en cola virtual y marca récord de demanda, ¿fue la más veloz?",economia,deportes
371,Alerta en Latinoamérica por películas que esconden malware y pueden dar acceso remoto a tu PC,ciencia_tecnologia,economia


Este F1 macro es el piso que el modelo ajustado tiene que superar.

## 2. Supervisión débil con un LLM

### 2.1 El modelo

Usamos la misma función `chat()` de la clase 1, que responde sin azar. Es decir, `do_sample=False`, lo que equivale a temperatura 0.

Primero se probó con Qwen2.5-0.5B-Instruct, el modelo de la clase 1. Con este mismo prompt coincidió solo en un 65% con la sección del diario y confundía mucho deportes con ciencia y tecnología, así que quedaba por debajo del zero-shot. Por eso se usa el modelo siguiente de la misma familia, **Qwen2.5-1.5B-Instruct**, con el mismo código. Para que entre en los 6 GB de la GPU se carga en `float16` y antes se libera la memoria del modelo zero-shot.

In [6]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# Liberamos la GPU del modelo zero-shot
del zero_shot
torch.cuda.empty_cache()

MODELO_LLM = "Qwen/Qwen2.5-1.5B-Instruct"
DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"
TIPO = torch.float16 if DISPOSITIVO == "cuda" else torch.float32

tok = AutoTokenizer.from_pretrained(MODELO_LLM)
llm = AutoModelForCausalLM.from_pretrained(MODELO_LLM, dtype=TIPO).to(DISPOSITIVO).eval()

def chat(mensaje, sistema=None, max_tokens=60):
    """Una respuesta del modelo, sin azar (greedy)."""
    msgs = ([{"role": "system", "content": sistema}] if sistema else []) + \
           [{"role": "user", "content": mensaje}]
    entrada = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True).to(DISPOSITIVO)
    with torch.no_grad():
        salida = llm.generate(**entrada, max_new_tokens=max_tokens, do_sample=False)
    return tok.decode(salida[0, entrada["input_ids"].shape[1]:], skip_special_tokens=True).strip()

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

C:\Users\Usuario\Documents\GitHub\Python-Intermedio\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuario\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

C:\Users\Usuario\Documents\GitHub\Python-Intermedio\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuario\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

### 2.2 El prompt

El prompt es estructurado y de lista cerrada. Da las 4 categorías numeradas y pide responder solo con el número, para que el modelo no invente una quinta. Le damos el titular y la bajada, porque para etiquetar conviene darle al LLM todo el contexto disponible.

In [7]:
SISTEMA = "Eres un editor que clasifica noticias peruanas. Respondes solo con un número."

OPCIONES = "1 = política\n2 = economía\n3 = deportes\n4 = ciencia y tecnología"

def armar_prompt(titular, bajada):
    return (f"Titular: {titular}\nBajada: {bajada}\n\n"
            f"¿A qué categoría pertenece esta noticia?\n{OPCIONES}\n\n"
            "Responde solo con el número.")

fila = noticias.iloc[0]
print(armar_prompt(fila["titular"], fila["bajada"]))
print("\nRespuesta de chat():", chat(armar_prompt(fila["titular"], fila["bajada"]), sistema=SISTEMA))

Titular: Keiko Fujimori ratifica su respaldo al ministro César Astudillo y al jefe de la PNP Fredy López
Bajada: Keiko Fujimori ratificó su respaldo al ministro de Defensa, César Astudillo, y al comandante general de la Policía Nacional del Perú (PNP), Fredy López, durante una ceremonia de ascenso de cinco suboficiales reconocidos por su destacada labor en la lucha contra la delincuencia.

¿A qué categoría pertenece esta noticia?
1 = política
2 = economía
3 = deportes
4 = ciencia y tecnología

Responde solo con el número.



Respuesta de chat(): 1


### 2.3 La confianza

La tarea pide una columna de `confianza`. La sacamos con la idea de la función `siguiente()` de la clase 1. En vez de generar texto, miramos la probabilidad que el modelo le da a cada token posible como siguiente palabra, y nos quedamos solo con los tokens `1`, `2`, `3` y `4`. Con `softmax` sobre esos cuatro puntajes obtenemos una probabilidad por categoría que suma 1. La categoría es la más probable y la confianza es su probabilidad.

Como no hay azar, la categoría elegida coincide con lo que respondería `chat()`.

In [8]:
IDS_NUMEROS = [tok.encode(str(n))[0] for n in range(1, 5)]
print("IDs de los tokens 1, 2, 3 y 4:", IDS_NUMEROS)

def clasificar_llm(titular, bajada):
    msgs = [{"role": "system", "content": SISTEMA},
            {"role": "user", "content": armar_prompt(titular, bajada)}]
    entrada = tok.apply_chat_template(msgs, add_generation_prompt=True,
                                      return_tensors="pt", return_dict=True).to(DISPOSITIVO)
    with torch.no_grad():
        logits = llm(**entrada).logits[0, -1]          # un puntaje por token del vocabulario
    probs = torch.softmax(logits[IDS_NUMEROS].float(), dim=-1)  # solo los 4 números, suman 1
    i = int(torch.argmax(probs))
    return ETIQUETAS[i], float(probs[i])

clasificar_llm(fila["titular"], fila["bajada"])

IDs de los tokens 1, 2, 3 y 4: [16, 17, 18, 19]


('politica', 0.9999892711639404)

### 2.4 Etiquetar todo el dataset

In [9]:
resultados = []
for titular, bajada in zip(noticias["titular"], noticias["bajada"]):
    resultados.append(clasificar_llm(titular, bajada))

etiquetas_llm = pd.DataFrame({
    "titular": noticias["titular"],
    "etiqueta_llm": [r[0] for r in resultados],
    "confianza": [round(r[1], 4) for r in resultados],
    "modelo_llm": f"{MODELO_LLM} (greedy, temperatura 0)",
    "fecha": str(date.today()),
})
etiquetas_llm.to_csv("datos/etiquetas_llm.csv", index=False, encoding="utf-8")
print(etiquetas_llm.shape)
etiquetas_llm.head()

(375, 5)


,titular,etiqueta_llm,confianza,modelo_llm,fecha
0,Keiko Fujimori ratifica su respaldo al ministro César Astudillo y al jefe de la PNP Fredy López,politica,1.0000,"Qwen/Qwen2.5-1.5B-Instruct (greedy, temperatura 0)",2026-10-14
1,Juan Sheput: “El Congreso ha apostado por la coyuntura y no por las reformas de fondo” | Entrevista,politica,1.0000,"Qwen/Qwen2.5-1.5B-Instruct (greedy, temperatura 0)",2026-10-14
2,"“Grupos hostiles”, el factor DINI y otras acciones: estos son los ocho puntos que incorpora el predictamen del Senado",politica,1.0000,"Qwen/Qwen2.5-1.5B-Instruct (greedy, temperatura 0)",2026-10-14
3,"Senadora Milagros Jáuregui a favor de analizar una segunda vuelta para la Alcaldía de Lima, pero no para el actual p...",politica,1.0000,"Qwen/Qwen2.5-1.5B-Instruct (greedy, temperatura 0)",2026-10-14
4,"Ley MAPE: Minem prevé presentar propuesta este jueves, ¿qué plantea y qué advierten los expertos sobre el Reinfo?",politica,0.9991,"Qwen/Qwen2.5-1.5B-Instruct (greedy, temperatura 0)",2026-10-14


La columna `fecha` es la fecha en que se etiquetó.

### 2.5 ¿Cuánto coincide el LLM con la sección del diario?

La etiqueta de la Parte 1 sale de la sección donde El Comercio publicó la nota. La comparamos con la del LLM.

In [10]:
noticias["etiqueta_llm"] = etiquetas_llm["etiqueta_llm"]
noticias["confianza_llm"] = etiquetas_llm["confianza"]

acuerdo = (noticias["etiqueta"] == noticias["etiqueta_llm"]).mean()
print(f"Coincidencia LLM vs sección del diario: {acuerdo:.1%}")
print(f"F1 macro del LLM tomando la sección como referencia: "
      f"{f1_score(noticias['etiqueta'], noticias['etiqueta_llm'], average='macro'):.3f}")

pd.crosstab(noticias["etiqueta"], noticias["etiqueta_llm"],
            rownames=["sección del diario"], colnames=["LLM"])

Coincidencia LLM vs sección del diario: 91.7%
F1 macro del LLM tomando la sección como referencia: 0.913


LLM,ciencia_tecnologia,deportes,economia,politica
sección del diario,,,,
ciencia_tecnologia,89,1,2,4
deportes,1,96,1,0
economia,9,4,66,9
politica,0,0,0,93


In [11]:
noticias.groupby(noticias["etiqueta"] == noticias["etiqueta_llm"])["confianza_llm"].describe()

,count,mean,std,min,25%,50%,75%,max
False,31.0,0.905939,0.134521,0.5466,0.85165,0.989,0.999,1.0
True,344.0,0.996271,0.021522,0.7178,0.99990,1.000,1.000,1.0


### 2.6 Auditoría humana de 20 casos

Se toman 20 noticias al azar con semilla fija y se revisa a mano si la etiqueta del LLM es correcta. La revisión se guarda en `datos/auditoria_20.csv`, en la columna `correcto` con 1 si el LLM acertó y 0 si no.

In [12]:
from pathlib import Path

ARCHIVO_AUDITORIA = Path("datos/auditoria_20.csv")
auditoria = noticias.sample(20, random_state=42)[["titular", "bajada", "etiqueta_llm", "confianza_llm"]]

if not ARCHIVO_AUDITORIA.exists():
    auditoria.assign(correcto="").to_csv(ARCHIVO_AUDITORIA, index_label="id", encoding="utf-8")
    print("Plantilla creada. Falta completar la columna 'correcto' a mano.")
auditoria

Plantilla creada. Falta completar la columna 'correcto' a mano.


,titular,bajada,etiqueta_llm,confianza_llm
167,"BTS en Lima: cómo 118 mil fans pueden mover hasta S/200 millones entre merchandising, restaurantes y más en el conc...","La llegada de BTS por primera vez al Perú no solo movilizará a sus fans, llamadas ‘Army’. La CCL estima entre S/150 ...",economia,0.9709
33,"Giannina, la impasible, una crónica de Fernando Vivas sobre la presidenta de la Comisión de Constitución de Diputados","La presidenta de la Comisión de Constitución, Giannina Avendaño, de Juntos Por el Perú, sacó de quicio al fujimorism...",politica,1.0000
15,"Juntos por el Perú impulsa propuestas que figuran en acuerdo firmado con el Fenatep, grupo vinculado al Movadef","Más de cinco propuestas legislativas, que figuran figuran en el acuerdo que el líder de Juntos por el Perú y excandi...",politica,1.0000
316,"Fallece Margaret Hamilton, pionera del programa Apolo y en la llegada del hombre a la Luna",Su trabajo es reconocido como inspiración para generaciones de informáticos e ingenieros. En 2016 el presidente Bara...,ciencia_tecnologia,1.0000
57,Keiko Fujimori expresa solidaridad con el papa León XIV y pide por su pronta recuperación: “Nuestras oraciones lo ac...,"La presidenta Keiko Fujimori expresó su solidaridad con el papa León XIV, quien será sometido a una intervención méd...",politica,1.0000
291,HomeView: Apple entra de lleno en el hogar con su primera pantalla inteligente con IA,"El equipo integra el procesador A18 y una versión renovada del asistente Siri, potenciada por Apple Intelligence. Sa...",ciencia_tecnologia,1.0000
76,Luis Galarreta ante Comisión de Constitución del Senado: “Dictamen aprobado por diputados no es suficiente”,"El presidente del Consejo de Ministros, Luis Galarreta, sustentó el pedido de facultades legislativas del Ejecutivo ...",politica,1.0000
119,"Eduardo Grisolle, CEO de 121: “El modelo de las agencias de holding está cumpliendo un ciclo,es el momento de las ag...","Tras más de dos décadas como CEO en el holding WPP, Grisolle analiza los cambios en la industria publicitaria. Ahora...",economia,0.9999
280,León XIV fue operado de cáncer de pulmón: qué implica la extirpación de un lóbulo y cómo es la recuperación,El pontífice fue operado con éxito de un cáncer de pulmón. Dos especialista explica en qué consiste la intervención ...,ciencia_tecnologia,0.9956
126,Precio del dólar en Perú: a cuánto cerró el tipo de cambio el martes 13 de octubre,"Precio del dólar hoy EN VIVO: cotización en compra y venta del dólar en el Perú en la jornada de hoy, martes 13 de o...",economia,0.9998
